In [ ]:
# codigo de validacao a parte
# esse codigo serve para analisar a equacao no formato txt e verificar as saidas dela,
# alem disso, conseguimos plotar os treinamentos ja feitos. (ainda em fase de teste)

import os
import gc
import dill
import numpy as np
import pandas as pd
import sympy as sp
import matplotlib.pyplot as plt
from deap import gp

from src.mggp import MGGP 

# =============================================================================
# 🚨 O TRUQUE: INICIALIZAR UM "MANEQUIM" PARA PREPARAR A MEMÓRIA DO DEAP
# =============================================================================
print("⚙️ Preparando o ambiente original da MGGP...")
u_dummy = np.zeros((10, 5)) 
y_dummy = np.zeros((10, 2))
dummy_mggp = MGGP(inputs=u_dummy, outputs=y_dummy, mode="MIMO")
# =============================================================================

# =============================================================================
# 🎛️ PAINEL DE CONFIGURAÇÃO 
# =============================================================================
DIRETORIO_BASE = r"C:\IC_alex\TreinamentoMGGP_2_0\RESULTADOS"
ID_TREINAMENTO = 8 
NUMERO_DO_MODELO = 8 
CAMINHO_VALIDACAO = r'C:\IC_alex\Database\wang22dv_bic_MGGP.xlsx'
OFFSET_Y = 10.0

# Tradutor: Mapeie aqui as variáveis do DEAP para a notação do seu artigo
TRADUTOR_VARIAVEIS = {
    "ARG0": "u1[k]",
    "ARG1": "u1[k-1]",
    "ARG2": "u2[k]",
    "ARG3": "u2[k-1]",
    "ARG4": "u3[k]",
    "ARG5": "u3[k-1]",
    "q10": "u10[k]",
    "q5": "u5[k]",
    "q2": "u2[k]"
    # Adicione mais conforme descobrir no seu dataset!
}

# =============================================================================
# FUNÇÕES MATEMÁTICAS E DE DADOS
# =============================================================================
def carregar_dados_teste(caminho_arquivo):
    df = pd.read_excel(caminho_arquivo, header=None)
    # Puxa as colunas de entrada (u) e saída (y)
    u = df.iloc[:, [14, 2, 22, 23, 11]].values
    y_raw = df.iloc[:, [18, 17]].values
    return u, y_raw + OFFSET_Y

def calcular_rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def formatar_estilo_gptips_multigene(lista_de_genes, idx_saida):
    """Lê TODOS os genes do MGGP, soma tudo e formata no estilo GPTIPS."""
    funcoes_locais = {
        'add': lambda x, y: x + y,
        'sub': lambda x, y: x - y,
        'subtraction': lambda x, y: x - y,
        'mul': lambda x, y: x * y,
        'div': lambda x, y: x / y
    }
    
    equacao_final_sympy = 0 # Começa vazia
    
    try:
        # Loop passando por todos os genes desse output
        for gene in lista_de_genes:
            texto_arvore = str(gp.PrimitiveTree(gene))
            equacao_gene = sp.sympify(texto_arvore, locals=funcoes_locais)
            equacao_final_sympy += equacao_gene # Soma o gene na equação principal
            
        # Expande e multiplica tudo
        equacao_expandida = sp.expand(equacao_final_sympy)
        
        termos = equacao_expandida.as_ordered_terms()
        constantes = [t for t in termos if t.is_constant()]
        variaveis = [t for t in termos if not t.is_constant()]
        
        print(f"Output {idx_saida + 1}:")
        bias = float(constantes[0]) if constantes else 0.0
        print(f"{bias:.5e} * ")
        
        for i, t in enumerate(variaveis):
            coef, var = t.as_coeff_Mul()
            var_str = str(var)
            
            for deap_var, real_var in TRADUTOR_VARIAVEIS.items():
                var_str = var_str.replace(deap_var, real_var)
                
            sinal = "+" if i < len(variaveis) - 1 else ""
            print(f"{float(coef):.5e} * {var_str} {sinal}")
            
    except Exception as e:
        print(f"Output {idx_saida + 1}:\n [Erro ao formatar a equação simbólica: {e}]")

# =============================================================================
# FUNÇÃO DO GRÁFICO
# =============================================================================
def plotar_validacao(yd, yp, rmse_vx, rmse_vy, offset):
    """Gera o gráfico comparando o Real vs Previsto e retira o offset visualmente."""
    print("\n📊 Gerando gráficos de validação...")
    
    # Remove o offset para o gráfico fazer sentido físico (Volta ao zero)
    yd_real = yd - offset
    yp_real = yp - offset
    
    fig, axs = plt.subplots(2, 1, figsize=(12, 8), dpi=100)
    fig.canvas.manager.set_window_title(f'Validação MGGP - Treinamento {ID_TREINAMENTO} | Modelo {NUMERO_DO_MODELO}')
    
    # --- Gráfico do Output 1 (Vx) ---
    axs[0].plot(yd_real[:, 0], label='Real (Vx)', color='black', linewidth=1.5)
    axs[0].plot(yp_real[:, 0], label='Previsto MGGP (Vx)', color='#ff7f0e', linestyle='--', linewidth=1.5)
    axs[0].set_title(f'Validação Vx - RMSE: {rmse_vx:.4f}', fontsize=12, fontweight='bold')
    axs[0].set_ylabel('Sinal')
    axs[0].legend(loc='upper right')
    axs[0].grid(True, linestyle=':', alpha=0.7)
    
    # --- Gráfico do Output 2 (Vy) ---
    axs[1].plot(yd_real[:, 1], label='Real (Vy)', color='black', linewidth=1.5)
    axs[1].plot(yp_real[:, 1], label='Previsto MGGP (Vy)', color='#1f77b4', linestyle='--', linewidth=1.5)
    axs[1].set_title(f'Validação Vy - RMSE: {rmse_vy:.4f}', fontsize=12, fontweight='bold')
    axs[1].set_xlabel('Amostras (k)')
    axs[1].set_ylabel('Sinal')
    axs[1].legend(loc='upper right')
    axs[1].grid(True, linestyle=':', alpha=0.7)
    
    plt.tight_layout()
    plt.show()

# =============================================================================
# EXECUÇÃO PRINCIPAL
# =============================================================================
if __name__ == "__main__":
    os.system('cls' if os.name == 'nt' else 'clear')
    print("=" * 60)
    print(f" INICIANDO LEITURA - TREINAMENTO {ID_TREINAMENTO} | MODELO {NUMERO_DO_MODELO}")
    print("=" * 60)

    pasta_alvo = os.path.join(DIRETORIO_BASE, f"Treinamento_{ID_TREINAMENTO}", "TRAINING")
    caminho_pkl = os.path.join(pasta_alvo, f"modelo_mape_{NUMERO_DO_MODELO}.pkl")
    
    if not os.path.exists(caminho_pkl):
        print(f"❌ Arquivo não encontrado: {caminho_pkl}")
    else:
        try:
            with open(caminho_pkl, 'rb') as f:
                modelo_carregado = dill.load(f)
            
            # --- EXTRAÇÃO MULTI-GENE ---
            print("\n🔍 EQUAÇÕES MATEMÁTICAS ENCONTRADAS (Pós-simplificação do SymPy):\n")
            for idx_saida, saida in enumerate(modelo_carregado):
                if len(saida) > 0:
                    formatar_estilo_gptips_multigene(saida, idx_saida)
            print("-" * 60)
            
            # --- VALIDAÇÃO E PLOT ---
            print(f"\n🔄 Rodando validação de teste usando RMSE...")
            u_test, y_test = carregar_dados_teste(CAMINHO_VALIDACAO)
            yp, yd = modelo_carregado.predict("FreeRun", y_test, u_test)
            
            min_len = min(len(yd), len(yp))
            yd, yp = yd[:min_len], yp[:min_len]
            
            erro_vx = calcular_rmse(yd[:, 0], yp[:, 0])
            erro_vy = calcular_rmse(yd[:, 1], yp[:, 1])
            
            print(f"📈 Resultados Finais -> RMSE Vx: {erro_vx:.4f} | RMSE Vy: {erro_vy:.4f}")
            
            plotar_validacao(yd, yp, erro_vx, erro_vy, OFFSET_Y)
            
        except Exception as e:
            print(f"\n❌ Erro durante a execução: {e}")